# Prompt Management: Versioning, A/B Testing, and Regression Testing

In production LLM applications, prompts are software artifacts. They need version control, testing, and deployment pipelines just like code. Treating prompts as mutable strings passed around in application code leads to the same problems as unversioned, untested code.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable
import json, hashlib, time

@dataclass
class PromptVersion:
    name: str
    version: str
    template: str
    variables: list
    created_at: float = field(default_factory=time.time)
    sha: str = ''
    tags: list = field(default_factory=list)

    def __post_init__(self):
        self.sha = hashlib.md5(self.template.encode()).hexdigest()[:8]

    def render(self, variables: dict) -> str:
        result = self.template
        missing = [v for v in self.variables if v not in variables]
        if missing:
            raise ValueError(f'Missing variables: {missing}')
        for k, v in variables.items():
            result = result.replace('{'+k+'}', str(v))
        return result

class PromptRegistry:
    def __init__(self):
        self.prompts: dict = {}  # name -> list of versions

    def register(self, prompt: PromptVersion):
        if prompt.name not in self.prompts:
            self.prompts[prompt.name] = []
        self.prompts[prompt.name].append(prompt)

    def get(self, name: str, version: str = 'latest') -> PromptVersion:
        if name not in self.prompts:
            raise KeyError(f'Prompt not found: {name}')
        versions = self.prompts[name]
        if version == 'latest':
            return versions[-1]
        matches = [v for v in versions if v.version == version]
        if not matches:
            raise KeyError(f'Version {version} not found for {name}')
        return matches[0]

    def run_regression(self, name: str, test_cases: list, model_fn: Callable) -> dict:
        results = {}
        for version in self.prompts.get(name, []):
            version_results = []
            for case in test_cases:
                try:
                    prompt = version.render(case['variables'])
                    output = model_fn(prompt)
                    passed = case['expected'].lower() in output.lower()
                    version_results.append({'case': case['name'], 'passed': passed})
                except Exception as e:
                    version_results.append({'case': case['name'], 'passed': False, 'error': str(e)})
            pass_rate = sum(r['passed'] for r in version_results) / len(version_results)
            results[version.version] = {'pass_rate': pass_rate, 'cases': version_results}
        return results

registry = PromptRegistry()
registry.register(PromptVersion(
    name='summarize',
    version='1.0.0',
    template='Summarize the following text in one sentence:\n\n{text}',
    variables=['text'],
    tags=['production'],
))
registry.register(PromptVersion(
    name='summarize',
    version='1.1.0',
    template='You are an expert at summarization. Provide a concise one-sentence summary of:\n\n{text}\n\nSummary:',
    variables=['text'],
    tags=['candidate'],
))

test_cases = [
    {'name': 'short_text', 'variables': {'text': 'The sky is blue because of Rayleigh scattering.'}, 'expected': 'blue'},
    {'name': 'technical', 'variables': {'text': 'Neural networks learn by adjusting weights via gradient descent.'}, 'expected': 'neural'},
]

def mock_model(prompt):
    if 'sky is blue' in prompt: return 'The sky appears blue due to light scattering.'
    if 'neural networks' in prompt: return 'Neural networks adjust weights using gradient descent.'
    return 'Summary of the text.'

results = registry.run_regression('summarize', test_cases, mock_model)
print('Prompt regression results:')
for version, r in results.items():
    print(f'  v{version}: {r["pass_rate"]:.0%} pass rate')
    for case in r['cases']:
        print(f'    {case["case"]}: {"PASS" if case["passed"] else "FAIL"}')

## A/B Testing Prompts

Prompt A/B testing routes a fraction of traffic to a candidate prompt and measures key metrics against the baseline:
1. Define success metrics (task completion rate, user satisfaction, refusal rate)
2. Route X% of requests to the candidate prompt
3. Measure metrics on both variants over a statistically significant sample
4. Use a statistical test (chi-squared for binary outcomes, t-test for continuous) to determine if the difference is significant
5. Roll out or reject the candidate based on results

Pitfalls: ensure the A/B split is truly random (not biased by time-of-day or user segment), log both the prompt version and all metrics for every request, and decide on success criteria before seeing data.